# ETL POC2 – Entreprises, entités, stagiaires et taxe d’apprentissage

## 1. Objectif du notebook

Ce notebook prépare la construction de l’ETL du POC2 à partir du fichier `Entreprises entites stagiaires.csv`.

L’objectif est de transformer l’export Aurion brut en données nettoyées et structurées, exploitables pour un chargement dans MongoDB puis pour un affichage dans CremeCRM.

Les traitements prévus sont :

- charger le fichier source ;
- supprimer les doublons stricts ;
- convertir les dates et les montants ;
- distinguer les entités, contacts, adresses, événements de taxe et informations de suivi pédagogique ;
- produire une structure de sortie propre pour la suite du POC2.

In [1]:
from pathlib import Path

import json
import pandas as pd

In [2]:
# Définition des chemins du projet

project_root = Path.cwd()

if project_root.name == "project_notebooks":
    project_root = project_root.parent

raw_file_path = project_root / "project_data" / "raw" / "Entreprises entites stagiaires.csv"

intermediate_dir = project_root / "project_data" / "intermediate"
processed_dir = project_root / "project_data" / "processed"
exports_dir = project_root / "project_data" / "exports"

print("Racine du projet :", project_root)
print("Fichier source :", raw_file_path)
print("Fichier source trouvé :", raw_file_path.exists())
print("Dossier intermediate trouvé :", intermediate_dir.exists())
print("Dossier processed trouvé :", processed_dir.exists())
print("Dossier exports trouvé :", exports_dir.exists())

Racine du projet : /home/carole/iadev/CRM
Fichier source : /home/carole/iadev/CRM/project_data/raw/Entreprises entites stagiaires.csv
Fichier source trouvé : True
Dossier intermediate trouvé : True
Dossier processed trouvé : True
Dossier exports trouvé : True


## Bloc 1 — Extraction, nettoyage et conversions de base

Ce premier bloc prépare les données avant leur structuration métier.

Il permet de charger l’export Aurion brut, de vérifier que les colonnes attendues sont présentes, puis de créer une table de travail nettoyée. Les traitements réalisés dans ce bloc concernent principalement :

* le chargement du fichier source ;
* la vérification des colonnes attendues ;
* la suppression des doublons stricts ;
* la suppression de la colonne technique d’export ;
* le renommage des colonnes pour faciliter l’ETL ;
* le nettoyage des champs texte ;
* la conversion des colonnes de comptage ;
* la conversion des dates d’événement ;
* la conversion des montants de taxe ;
* la conversion de l’assujettissement à la taxe ;
* le contrôle global de la table nettoyée.

À la fin de ce bloc, la table `df_work` constitue la base de travail propre pour la suite de l’ETL.


### 2. Chargement du fichier source

Le fichier source contient un préambule Aurion de 21 lignes.  
Ces lignes sont ignorées afin de charger uniquement le tableau de données métier.

In [5]:
# Chargement du fichier CSV source

df_raw = pd.read_csv(
    raw_file_path,
    sep=";",
    encoding="cp1252",
    encoding_errors="replace",
    skiprows=21
)

# Vérification du chargement
assert df_raw.shape[0] > 0
assert df_raw.shape[1] == 24

f"Chargement OK - {df_raw.shape[0]} lignes et {df_raw.shape[1]} colonnes"


'Chargement OK - 13345 lignes et 24 colonnes'

### 3. Vérification des colonnes attendues

Cette étape vérifie que le fichier chargé contient bien les colonnes attendues avant de commencer les transformations.

In [6]:
# Vérification des colonnes attendues dans le fichier source

expected_columns = [
    "Unnamed: 0",
    "Code.Entité",
    "Libellé.Entité",
    "Code.Type d'entité",
    "Assujetti.Entité",
    "Rue (ligne 1).Adresse",
    "Rue (ligne 2).Adresse",
    "Rue (ligne 3).Adresse",
    "Rue (ligne 4).Adresse",
    "Code postal.Ville",
    "Nom.Ville",
    "Code.Type d'adresse",
    "Nombre de stage (sans contrat pro)",
    "Noms des stagiaires",
    "Nombre de contrat pro",
    "Noms des alternants contrats pro",
    "Nombre apprentissage",
    "Noms des apprentis",
    "Code.Type d'événement",
    "Montant global.Taxe versement",
    "Début.Événement",
    "Code.Fonctions",
    "Nom.Individu",
    "Prénom.Individu",
]

missing_columns = [column for column in expected_columns if column not in df_raw.columns]
unexpected_columns = [column for column in df_raw.columns if column not in expected_columns]

assert not missing_columns, f"Colonnes manquantes : {missing_columns}"
assert not unexpected_columns, f"Colonnes inattendues : {unexpected_columns}"

f"Colonnes OK - {len(df_raw.columns)} colonnes attendues trouvées"

'Colonnes OK - 24 colonnes attendues trouvées'

### 4. Création d’une copie de travail et suppression des doublons stricts

Cette étape crée une copie du fichier chargé afin de conserver `df_raw` comme référence brute.

Les doublons stricts sont ensuite supprimés après exclusion de la colonne technique `Unnamed: 0`, qui correspond à l’index de l’export Aurion.

In [7]:
# Création d'une copie de travail

df_work = df_raw.copy()

# Suppression des doublons stricts après exclusion de la colonne technique
columns_for_duplicates = df_work.columns.difference(["Unnamed: 0"])

initial_rows = len(df_work)
duplicates_count = df_work.duplicated(subset=columns_for_duplicates).sum()

df_work = df_work.drop_duplicates(
    subset=columns_for_duplicates,
    keep="first"
).copy()

removed_rows = initial_rows - len(df_work)

# Vérification de la suppression
assert removed_rows == duplicates_count
assert len(df_work) < initial_rows

f"Doublons stricts supprimés - {removed_rows} lignes supprimées, {len(df_work)} lignes conservées"

'Doublons stricts supprimés - 910 lignes supprimées, 12435 lignes conservées'

### 5. Suppression de la colonne technique d’export

La colonne `Unnamed: 0` correspond à l’index technique généré par l’export Aurion.  
Elle n’est pas une donnée métier et peut être supprimée de la copie de travail.

In [8]:
# Suppression de la colonne technique d'export

df_work = df_work.drop(columns=["Unnamed: 0"], errors="ignore")

# Vérification de la suppression
assert "Unnamed: 0" not in df_work.columns
assert df_work.shape[0] == 12435

f"Colonne technique supprimée - {df_work.shape[0]} lignes et {df_work.shape[1]} colonnes conservées"

'Colonne technique supprimée - 12435 lignes et 23 colonnes conservées'

### 6. Renommage des colonnes pour l’ETL

Les colonnes issues de l’export Aurion sont renommées avec des noms plus simples afin de faciliter les traitements ETL.

Les noms d’origine restent documentés dans l’EDA. Dans ce notebook ETL, les noms simplifiés permettent d’éviter les erreurs liées aux espaces, accents, apostrophes ou parenthèses.

In [9]:
# Renommage des colonnes pour faciliter les traitements ETL

column_mapping = {
    "Code.Entité": "code_entite",
    "Libellé.Entité": "libelle_entite",
    "Code.Type d'entité": "type_entite",
    "Assujetti.Entité": "assujetti_entite",
    "Rue (ligne 1).Adresse": "adresse_ligne_1",
    "Rue (ligne 2).Adresse": "adresse_ligne_2",
    "Rue (ligne 3).Adresse": "adresse_ligne_3",
    "Rue (ligne 4).Adresse": "adresse_ligne_4",
    "Code postal.Ville": "code_postal",
    "Nom.Ville": "ville",
    "Code.Type d'adresse": "type_adresse",
    "Nombre de stage (sans contrat pro)": "nombre_stages",
    "Noms des stagiaires": "noms_stagiaires",
    "Nombre de contrat pro": "nombre_contrats_pro",
    "Noms des alternants contrats pro": "noms_alternants_contrats_pro",
    "Nombre apprentissage": "nombre_apprentissages",
    "Noms des apprentis": "noms_apprentis",
    "Code.Type d'événement": "type_evenement",
    "Montant global.Taxe versement": "montant_taxe",
    "Début.Événement": "debut_evenement",
    "Code.Fonctions": "fonction_contact",
    "Nom.Individu": "nom_contact",
    "Prénom.Individu": "prenom_contact",
}

df_work = df_work.rename(columns=column_mapping)

# Vérification du renommage
assert len(df_work.columns) == 23
assert "code_entite" in df_work.columns
assert "montant_taxe" in df_work.columns
assert "debut_evenement" in df_work.columns

f"Renommage OK - {len(df_work.columns)} colonnes prêtes pour l'ETL"

"Renommage OK - 23 colonnes prêtes pour l'ETL"

### 7. Nettoyage des colonnes texte

Les colonnes textuelles sont harmonisées afin de supprimer les espaces inutiles en début ou fin de valeur.

Les chaînes vides sont également remplacées par des valeurs manquantes, afin de faciliter les traitements suivants.

In [10]:
# Nettoyage des colonnes texte

text_columns = df_work.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    df_work[column] = df_work[column].astype("string").str.strip()
    df_work[column] = df_work[column].replace("", pd.NA)

# Vérification du nettoyage
empty_strings_count = (df_work[text_columns] == "").sum().sum()

assert empty_strings_count == 0

f"Nettoyage texte OK - {len(text_columns)} colonnes texte harmonisées"

'Nettoyage texte OK - 20 colonnes texte harmonisées'

### 8. Conversion des colonnes de comptage

Les colonnes liées au nombre de stages, contrats professionnels et apprentissages sont converties en entiers.

Les valeurs manquantes sont conservées afin de ne pas transformer automatiquement une absence d’information en valeur zéro.

In [11]:
# Conversion des colonnes de comptage en entiers nullable

count_columns = [
    "nombre_stages",
    "nombre_contrats_pro",
    "nombre_apprentissages",
]

for column in count_columns:
    df_work[column] = pd.to_numeric(df_work[column], errors="coerce")
    
    non_missing_values = df_work[column].dropna()
    assert (non_missing_values % 1 == 0).all(), f"Valeurs non entières détectées dans {column}"
    
    df_work[column] = df_work[column].astype("Int64")

# Vérification de la conversion
for column in count_columns:
    assert str(df_work[column].dtype) == "Int64"

f"Colonnes de comptage OK - {len(count_columns)} colonnes converties en entiers nullable"

'Colonnes de comptage OK - 3 colonnes converties en entiers nullable'

### 9. Conversion des dates d’événement

La colonne `debut_evenement` est convertie en date afin de préparer l’exploitation des événements de taxe d’apprentissage.

Une colonne `date_evenement` est créée pour conserver la date convertie, puis une colonne `annee_evenement` est ajoutée pour faciliter les regroupements par année.

In [12]:
# Conversion des dates d'événement

date_with_time = pd.to_datetime(
    df_work["debut_evenement"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

date_without_time = pd.to_datetime(
    df_work["debut_evenement"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_work["date_evenement"] = date_with_time.fillna(date_without_time)
df_work["annee_evenement"] = df_work["date_evenement"].dt.year.astype("Int64")

# Vérification de la conversion
rows_with_event_date = df_work["debut_evenement"].notna()
invalid_dates_count = (
    rows_with_event_date
    & df_work["date_evenement"].isna()
).sum()

assert invalid_dates_count == 0

f"Dates OK - {df_work['date_evenement'].notna().sum()} dates converties"

'Dates OK - 4284 dates converties'

### 10. Conversion des montants de taxe

La colonne `montant_taxe` est convertie en nombre décimal afin de permettre les contrôles, les calculs et la création des événements de taxe.

Les valeurs manquantes sont conservées afin de ne pas créer de montant artificiel.

In [13]:
# Conversion des montants de taxe en nombres décimaux

montant_taxe_clean = (
    df_work["montant_taxe"]
    .astype("string")
    .str.replace(",", ".", regex=False)
    .str.replace(r"\s+", "", regex=True)
)

df_work["montant_taxe_decimal"] = pd.to_numeric(
    montant_taxe_clean,
    errors="coerce"
)

# Vérification de la conversion
rows_with_montant = df_work["montant_taxe"].notna()
invalid_montants_count = (
    rows_with_montant
    & df_work["montant_taxe_decimal"].isna()
).sum()

assert invalid_montants_count == 0

f"Montants OK - {df_work['montant_taxe_decimal'].notna().sum()} montants convertis"

'Montants OK - 4284 montants convertis'

### 11. Conversion de l’assujettissement à la taxe

La colonne `assujetti_entite` est convertie en booléen nullable.

La valeur `Vrai` devient `True`.  
Les valeurs manquantes sont conservées afin de ne pas interpréter une absence d’information comme un `False`.

In [14]:
# Conversion de l'assujettissement en booléen nullable

known_assujetti_values = df_work["assujetti_entite"].dropna().unique().tolist()

assert set(known_assujetti_values) == {"Vrai"}, (
    f"Valeurs inattendues dans assujetti_entite : {known_assujetti_values}"
)

df_work["assujetti_entite_bool"] = df_work["assujetti_entite"].map({
    "Vrai": True
}).astype("boolean")

# Vérification de la conversion
assert df_work["assujetti_entite_bool"].sum() == df_work["assujetti_entite"].eq("Vrai").sum()
assert df_work["assujetti_entite_bool"].isna().sum() == df_work["assujetti_entite"].isna().sum()

f"Assujettissement OK - {df_work['assujetti_entite_bool'].sum()} valeurs True, {df_work['assujetti_entite_bool'].isna().sum()} valeurs non renseignées"

'Assujettissement OK - 11196 valeurs True, 1239 valeurs non renseignées'

### 12. Contrôle global de la table nettoyée

Cette étape vérifie que la table de travail est cohérente après les premiers traitements : suppression des doublons stricts, suppression de la colonne technique, renommage des colonnes et conversions principales.

In [15]:
# Contrôle global de la table de travail nettoyée

remaining_duplicates = df_work.duplicated().sum()

assert remaining_duplicates == 0
assert "code_entite" in df_work.columns
assert "date_evenement" in df_work.columns
assert "montant_taxe_decimal" in df_work.columns
assert "assujetti_entite_bool" in df_work.columns
assert len(df_work) == 12435

f"Table nettoyée OK - {len(df_work)} lignes, {len(df_work.columns)} colonnes, aucun doublon strict restant"

'Table nettoyée OK - 12435 lignes, 27 colonnes, aucun doublon strict restant'

## Bloc 2 — Structuration métier des données

Les étapes précédentes ont permis de charger le fichier source, de supprimer les doublons stricts, de retirer la colonne technique, de renommer les colonnes et de convertir les principaux champs nécessaires à l’ETL.

La suite du notebook consiste à structurer les données selon les objets métier utiles au POC2 :

* les entités / organisations ;
* les contacts ;
* les adresses ;
* les événements de taxe ;
* les informations de suivi pédagogique.

Cette structuration permettra ensuite de préparer les exports JSON et le chargement dans MongoDB.


### 13. Création de la table des entités

Cette étape prépare une table des entités uniques à partir du champ `code_entite`.

Chaque entité doit être créée une seule fois, même si elle apparaît plusieurs fois dans l’export Aurion à cause de plusieurs contacts, adresses ou événements associés.

In [16]:
# Création de la table des entités uniques

assert df_work["code_entite"].notna().all(), "Des lignes sans code_entite sont présentes"

def first_valid_value(series):
    values = series.dropna()
    if values.empty:
        return pd.NA
    return values.iloc[0]

df_entites = (
    df_work
    .groupby("code_entite", as_index=False)
    .agg({
        "libelle_entite": first_valid_value,
        "type_entite": first_valid_value,
        "assujetti_entite_bool": first_valid_value,
    })
)

# Vérification de la table des entités
assert df_entites["code_entite"].is_unique
assert len(df_entites) == df_work["code_entite"].nunique()

f"Entités OK - {len(df_entites)} entités uniques créées"

'Entités OK - 6738 entités uniques créées'

### 14. Création de la table des contacts

Cette étape prépare une table des contacts à partir des lignes contenant au moins une information nominative exploitable.

Un contact est rattaché à une entité grâce au champ `code_entite`. Les lignes sans nom ni prénom ne sont pas utilisées pour créer un contact, car elles ne permettent pas d’identifier une personne.


In [17]:
# Création de la table des contacts

contact_rows = df_work[
    df_work[["nom_contact", "prenom_contact"]]
    .notna()
    .any(axis=1)
].copy()

df_contacts = (
    contact_rows[
        [
            "code_entite",
            "libelle_entite",
            "nom_contact",
            "prenom_contact",
            "fonction_contact",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=["code_entite", "nom_contact", "prenom_contact", "fonction_contact"],
        na_position="last"
    )
    .reset_index(drop=True)
)

df_contacts.insert(
    0,
    "contact_id",
    ["contact_" + str(index + 1).zfill(6) for index in range(len(df_contacts))]
)

# Vérification de la table des contacts
assert df_contacts["contact_id"].is_unique
assert df_contacts["code_entite"].notna().all()
assert df_contacts[["nom_contact", "prenom_contact"]].notna().any(axis=1).all()

f"Contacts OK - {len(df_contacts)} contacts distincts créés"

'Contacts OK - 3982 contacts distincts créés'

### 15. Création de la table des adresses

Cette étape prépare une table des adresses distinctes rattachées aux entités.

Une même entité peut avoir plusieurs adresses selon le contexte métier : adresse principale, adresse de facturation, adresse secondaire ou adresse liée à l’alternance et aux stages.

Les lignes sont conservées lorsqu’au moins une information d’adresse est présente.

In [18]:
# Création de la table des adresses

address_columns = [
    "code_entite",
    "libelle_entite",
    "type_adresse",
    "adresse_ligne_1",
    "adresse_ligne_2",
    "adresse_ligne_3",
    "adresse_ligne_4",
    "code_postal",
    "ville",
]

address_info_columns = [
    "type_adresse",
    "adresse_ligne_1",
    "adresse_ligne_2",
    "adresse_ligne_3",
    "adresse_ligne_4",
    "code_postal",
    "ville",
]

address_rows = df_work[
    df_work[address_info_columns]
    .notna()
    .any(axis=1)
].copy()

df_adresses = (
    address_rows[address_columns]
    .drop_duplicates()
    .sort_values(
        by=["code_entite", "type_adresse", "adresse_ligne_1", "code_postal", "ville"],
        na_position="last"
    )
    .reset_index(drop=True)
)

df_adresses.insert(
    0,
    "adresse_id",
    ["adresse_" + str(index + 1).zfill(6) for index in range(len(df_adresses))]
)

# Vérification de la table des adresses
assert df_adresses["adresse_id"].is_unique
assert df_adresses["code_entite"].notna().all()
assert df_adresses[address_info_columns].notna().any(axis=1).all()

f"Adresses OK - {len(df_adresses)} adresses distinctes créées"

'Adresses OK - 7446 adresses distinctes créées'

### 16. Création de la table des événements de taxe

Cette étape prépare une table des événements de taxe d’apprentissage.

Les événements sont dédupliqués à partir d’une clé métier indicative composée de l’entité, de la date de l’événement et du montant du versement. Cette règle permet de limiter les répétitions liées à la structure plate de l’export Aurion.

In [19]:
# Création de la table des événements de taxe

taxe_rows = df_work[
    df_work["type_evenement"].eq("TAXE_VERSEMENT_ENTREPRISE")
].copy()

# Vérification des informations nécessaires aux événements de taxe
assert taxe_rows["code_entite"].notna().all()
assert taxe_rows["date_evenement"].notna().all()
assert taxe_rows["montant_taxe_decimal"].notna().all()

taxe_key_columns = [
    "code_entite",
    "date_evenement",
    "montant_taxe_decimal",
]

df_taxe_events = (
    taxe_rows
    .groupby(taxe_key_columns, as_index=False)
    .agg({
        "libelle_entite": first_valid_value,
        "type_evenement": first_valid_value,
        "montant_taxe": first_valid_value,
        "annee_evenement": first_valid_value,
    })
    .sort_values(
        by=["code_entite", "date_evenement", "montant_taxe_decimal"]
    )
    .reset_index(drop=True)
)

df_taxe_events.insert(
    0,
    "taxe_event_id",
    ["taxe_" + str(index + 1).zfill(6) for index in range(len(df_taxe_events))]
)

# Vérification de la table des événements de taxe
assert df_taxe_events["taxe_event_id"].is_unique
assert df_taxe_events[taxe_key_columns].duplicated().sum() == 0
assert len(df_taxe_events) <= len(taxe_rows)

f"Événements taxe OK - {len(df_taxe_events)} événements distincts créés à partir de {len(taxe_rows)} lignes taxe"

'Événements taxe OK - 1293 événements distincts créés à partir de 4284 lignes taxe'

### 17. Création de la table de suivi pédagogique

Cette étape prépare une table dédiée aux informations de suivi pédagogique : stages, contrats professionnels et apprentissages.

Les lignes sont conservées lorsqu’au moins une information de suivi est présente. Les valeurs manquantes sont conservées afin de ne pas transformer automatiquement une absence d’information en zéro.

In [20]:
# Création de la table de suivi pédagogique

suivi_columns = [
    "code_entite",
    "libelle_entite",
    "nombre_stages",
    "noms_stagiaires",
    "nombre_contrats_pro",
    "noms_alternants_contrats_pro",
    "nombre_apprentissages",
    "noms_apprentis",
]

suivi_info_columns = [
    "nombre_stages",
    "noms_stagiaires",
    "nombre_contrats_pro",
    "noms_alternants_contrats_pro",
    "nombre_apprentissages",
    "noms_apprentis",
]

suivi_rows = df_work[
    df_work[suivi_info_columns]
    .notna()
    .any(axis=1)
].copy()

df_suivi_pedagogique = (
    suivi_rows[suivi_columns]
    .drop_duplicates()
    .sort_values(by=["code_entite", "libelle_entite"])
    .reset_index(drop=True)
)

df_suivi_pedagogique.insert(
    0,
    "suivi_id",
    ["suivi_" + str(index + 1).zfill(6) for index in range(len(df_suivi_pedagogique))]
)

# Vérification de la table de suivi pédagogique
assert df_suivi_pedagogique["suivi_id"].is_unique
assert df_suivi_pedagogique["code_entite"].notna().all()
assert df_suivi_pedagogique[suivi_info_columns].notna().any(axis=1).all()

f"Suivi pédagogique OK - {len(df_suivi_pedagogique)} lignes distinctes créées"

'Suivi pédagogique OK - 1111 lignes distinctes créées'

### 18. Contrôle de cohérence des tables structurées

Cette étape vérifie que les tables créées à partir du fichier nettoyé sont cohérentes entre elles.

Les contacts, adresses, événements de taxe et lignes de suivi pédagogique doivent tous être rattachés à une entité existante grâce au champ `code_entite`.


In [22]:
# Contrôle de cohérence entre les tables structurées

codes_entites = set(df_entites["code_entite"])

contacts_invalides = set(df_contacts["code_entite"]) - codes_entites
adresses_invalides = set(df_adresses["code_entite"]) - codes_entites
taxe_invalides = set(df_taxe_events["code_entite"]) - codes_entites
suivi_invalides = set(df_suivi_pedagogique["code_entite"]) - codes_entites

assert not contacts_invalides, f"Contacts rattachés à des entités inconnues : {contacts_invalides}"
assert not adresses_invalides, f"Adresses rattachées à des entités inconnues : {adresses_invalides}"
assert not taxe_invalides, f"Événements taxe rattachés à des entités inconnues : {taxe_invalides}"
assert not suivi_invalides, f"Suivis pédagogiques rattachés à des entités inconnues : {suivi_invalides}"

f"Cohérence OK - {len(df_entites)} entités servent de référence aux tables structurées"

'Cohérence OK - 6738 entités servent de référence aux tables structurées'

## Bloc 3 — Export des données structurées

Ce bloc prépare les fichiers JSON issus de l’ETL.

Les tables créées dans le notebook sont exportées afin de pouvoir être utilisées ensuite pour le chargement dans MongoDB, puis pour l’intégration ou l’affichage dans CremeCRM.

### 19. Export des tables structurées en JSON

Cette étape exporte les principales tables métier dans le dossier `project_data/processed`.

Chaque fichier JSON correspond à un objet métier préparé par l’ETL : entités, contacts, adresses, événements de taxe et suivi pédagogique.

In [23]:
# Export des tables structurées en fichiers JSON

json_exports = {
    "poc2_entites.json": df_entites,
    "poc2_contacts.json": df_contacts,
    "poc2_adresses.json": df_adresses,
    "poc2_taxe_events.json": df_taxe_events,
    "poc2_suivi_pedagogique.json": df_suivi_pedagogique,
}

for file_name, dataframe in json_exports.items():
    output_path = processed_dir / file_name
    
    dataframe.to_json(
        output_path,
        orient="records",
        force_ascii=False,
        indent=2,
        date_format="iso"
    )
    
    assert output_path.exists(), f"Fichier non créé : {output_path}"

f"Exports JSON OK - {len(json_exports)} fichiers créés dans {processed_dir}"

'Exports JSON OK - 5 fichiers créés dans /home/carole/iadev/CRM/project_data/processed'

### 20. Vérification des fichiers JSON générés

Cette étape relit les fichiers JSON exportés afin de vérifier qu’ils sont bien exploitables et que le nombre d’objets correspond aux tables structurées.

In [24]:
# Vérification des fichiers JSON exportés

expected_counts = {
    "poc2_entites.json": len(df_entites),
    "poc2_contacts.json": len(df_contacts),
    "poc2_adresses.json": len(df_adresses),
    "poc2_taxe_events.json": len(df_taxe_events),
    "poc2_suivi_pedagogique.json": len(df_suivi_pedagogique),
}

for file_name, expected_count in expected_counts.items():
    file_path = processed_dir / file_name
    
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    assert isinstance(data, list), f"Le fichier {file_name} ne contient pas une liste JSON"
    assert len(data) == expected_count, (
        f"Nombre d'objets incorrect dans {file_name} : "
        f"{len(data)} au lieu de {expected_count}"
    )

f"Vérification JSON OK - {len(expected_counts)} fichiers relus avec succès"

'Vérification JSON OK - 5 fichiers relus avec succès'

### 21. Synthèse des volumes produits par l’ETL

Cette étape récapitule les volumes produits à partir de l’export Aurion nettoyé.

Elle permet de vérifier rapidement le nombre d’objets métier préparés pour la suite du POC2.


In [25]:
# Synthèse des volumes produits par l'ETL

etl_summary = {
    "lignes_source_brutes": len(df_raw),
    "lignes_apres_nettoyage": len(df_work),
    "entites": len(df_entites),
    "contacts": len(df_contacts),
    "adresses": len(df_adresses),
    "evenements_taxe": len(df_taxe_events),
    "suivi_pedagogique": len(df_suivi_pedagogique),
}

for label, count in etl_summary.items():
    print(f"{label} : {count}")

assert etl_summary["lignes_source_brutes"] == 13345
assert etl_summary["lignes_apres_nettoyage"] == 12435
assert etl_summary["entites"] == 6738

"Synthèse ETL OK"

lignes_source_brutes : 13345
lignes_apres_nettoyage : 12435
entites : 6738
contacts : 3982
adresses : 7446
evenements_taxe : 1293
suivi_pedagogique : 1111


'Synthèse ETL OK'

## Conclusion du notebook ETL

Ce notebook a permis de préparer une première version structurée de l’ETL POC2 à partir du fichier `Entreprises entites stagiaires.csv`.

Les traitements réalisés ont permis de :

* charger l’export Aurion en ignorant le préambule ;
* vérifier les colonnes attendues ;
* supprimer les doublons stricts ;
* retirer la colonne technique d’export ;
* renommer les colonnes pour faciliter les traitements ;
* nettoyer les champs texte ;
* convertir les colonnes de comptage, les dates, les montants de taxe et l’assujettissement ;
* créer des tables métier distinctes pour les entités, contacts, adresses, événements de taxe et informations de suivi pédagogique ;
* exporter les données structurées en fichiers JSON exploitables.

Les fichiers JSON produits dans `project_data/processed` constituent la base de travail pour la prochaine étape : le chargement dans MongoDB, puis la préparation de l’affichage dans CremeCRM.

La logique validée dans ce notebook pourra ensuite être transformée en script Python réexécutable dans `project_scripts/ETL/ETL_poc2.py`.
